In [2]:
import os
os.chdir("/content/drive/MyDrive/kisco/Data")

In [3]:
import pandas as pd
import numpy as np

csv_data = pd.read_csv("wine.csv")
csv_data.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [4]:
wine_data = csv_data[["alcohol", "sugar", "pH"]]
wine_target = csv_data["class"]

In [5]:
from sklearn.model_selection import train_test_split
train_data, test_data, train_target, test_target = train_test_split(wine_data, wine_target, test_size=0.2, random_state=42)

###랜덤포레스트
![](https://www.tibco.com/sites/tibco/files/media_entity/2021-05/random-forest-diagram.svg)

랜덤 포레스트는 기능을 무작위로 선택하고 관찰하여 의사 결정 트리의 포레스트를 만든 다음 결과를 평균화합니다.

* 랜덤 포레스트의 이점

> 상대적 중요성을 측정하기 쉬움
해당 기능을 사용하여 해당 포레스트에 있는 모든 트리의 불순물을 줄이는 노드를 보면 기능의 중요성을 쉽게 측정할 수 있습니다.

* 다재다능

> 랜덤 포레스트는 분류 및 회귀 작업 모두에 사용할 수 있기 때문에 매우 다재다능합니다.

* 과적합 없음

> 포레스트에 트리가 충분하면 과적합의 위험이 거의 또는 전혀 없습니다.

* 높은 정확도

> 하위 그룹 간에 상당한 차이가 있는 여러 트리를 사용하면 랜덤 포레스트가 매우 정확한 예측 도구가 됩니다.

In [6]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(rf_model, train_data, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.9973541965122431 0.8905151032797809


In [7]:
rf_model.fit(train_data, train_target)
print(rf_model.feature_importances_)

[0.23167441 0.50039841 0.26792718]


In [8]:
rf_model = RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=42)
rf_model.fit(train_data, train_target)
print(rf_model.oob_score_)

0.8934000384837406


###엑스트라 트리(extra trees)
* 엑스트라 트리는 랜덤 포레스트와 매우 비슷하게 동작
* 기본적으로 100개의 결정 트리를 훈련
* 전체 특성 중 일부 특성을 랜덤하게 선택해 노드를 분할
* 랜덤 포레스트와 엑스트라 트리의 차이점
* > 부투스트랩 샘플을 사용하지 않는다는 점
* > 각 결정 트리를 만들 때 전체 훈련 세트를 사용
* > 노드를 분할 할 때 가장 좋은 분할을 찾는 것이 아니라 무작위로 분할

* 하나의 결정 트리에서 특성을 무작위로 분할한다면 성능이 낮아지겠지만 많은 트리를 앙상블 하기 때문에 과대적합을 막고 검증 세트의 점수를 높이는 효과
* 보통 엑스트라 트리가 무작위성이 좀 더 커 랜덤 포레스트보다 더 많은 결정 트리를 훈련해야 함
* 랜덤하게 노드를 분할하기 때문에 빠른 계산 속도가 엑스트라 트리의 장점



In [9]:
from sklearn.ensemble import ExtraTreesClassifier

et_model = ExtraTreesClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(et_model, train_data, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.9974503966084433 0.8887848893166506


In [10]:
et_model.fit(train_data, train_target)
print(et_model.feature_importances_)

[0.20183568 0.52242907 0.27573525]


###그레이디언트 부스팅(gradient boosting)
* 그레이디언트 부스팅은 깊이가 얕은 결정 트리를 사용하여 이전 트리의 오차를 보완하는 방식

* 사이킷런의 GradientBoostingClassifier는 기본적으로 깊이가 3인 결정 트리를 100개 사용

* 깊이가 얕은 결정 트리를 사용하기 때문에 과대적합에 강하고 일반적으로 높은 일반화 성능을 기대

* 그레이디언트 부스팅은 경사 하강법을 사용. 분류에서는 로지스틱 손실 함수를, 회귀에서는 평균 제곱 오차 함수를 사용

In [11]:
from sklearn.ensemble import GradientBoostingClassifier

gs_model = GradientBoostingClassifier(random_state=42)
scores = cross_validate(gs_model, train_data, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.8881086892152563 0.8720430147331015


In [12]:
gs_model = GradientBoostingClassifier(n_estimators=500, max_depth=1, random_state=42)
scores = cross_validate(gs_model, train_data, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.8720898805575255 0.8664642407640482


In [13]:
gs_model = GradientBoostingClassifier(n_estimators=500, learning_rate=0.2, random_state=42)
scores = cross_validate(gs_model, train_data, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.9464595437171814 0.8780082549788999


In [14]:
gs_model.fit(train_data, train_target)
print(gs_model.feature_importances_)

[0.15872278 0.68010884 0.16116839]


### 히스토그램 기반 그레이디언트 부스팅(histogram-based gradient boosting)
* 히스토그램 기반 그레이디언트 부스팅은 정형 데이터를 다루는 머신러닝 알고리즘 중에 가장 인기가 높은 알고리즘

* 히스토그램 기반 그레이디언트 부스팅은 먼저 입력 특성을 256개의 구간으로 나눈다. 따라서 노드를 분할할 때 최적의 분할을 매우 빠르게 찾을 수 있다.

* 히스토그램 기반 그레이디언트 부스팅은 256개의 구간 중에서 하나를 떼어 놓고 누락된 값을 위해서 사용. 따라서 입력에 누락된 특성이 있어도 이를 따라 전처리할 필요가 없다.

In [16]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_model = HistGradientBoostingClassifier(random_state=42)
scores = cross_validate(hgb_model, train_data, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.9321723946453317 0.8801241948619236


In [18]:
from sklearn.inspection import permutation_importance
hgb_model.fit(train_data, train_target)
result = permutation_importance(hgb_model, train_data, train_target, n_repeats=10, random_state=42)
print(result.importances_mean)

[0.08876275 0.23438522 0.08027708]


In [ ]:
from lightgbm import LGBMClassifier
model = LGBMClassifier(random_state=42)
scores = cross_validate(model, train_data, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.9338079582727165 0.8789710890649293


In [ ]:
from xgboost import XGBClassifier
model = XGBClassifier(tree_method="hist", trandom_state=42)
scores = cross_validate(model, train_data, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores["train_score"]), np.mean(scores["test_score"]))

0.8824322471423747 0.8726214185237284
